In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.preprocessing import convert_dtypes

In [4]:
product_features = pd.read_csv(r"C:\Users\Lenovo\Customer-Intelligence-Platform\data\processed\product_summary_clean.csv")

In [8]:
product_features.head(5)

,product_id,product_category_name,total_units_sold,total_orders,total_revenue,total_freight,avg_price,avg_review_score,avg_delivery_days
0,00066f42aeeb9f3007548bb9d3f33c38,perfumaria,1,1,101.65,18.59,101.65,5.0,17.0
1,00088930e925c41fd95ebfe695fd2655,automotivo,1,1,129.90,13.93,129.90,4.0,11.0
2,0009406fd7479715e4bef61dd91f2462,cama_mesa_banho,1,1,229.00,13.10,229.00,1.0,16.0
3,000b8f95fcb9e0096488278317764d19,utilidades_domesticas,2,2,117.80,39.20,58.90,5.0,6.5
4,000d9be29b5207b54e86aa1b1ac54872,relogios_presentes,1,1,199.00,19.27,199.00,5.0,7.0


### 1. Revenue per order
Revenue per Order= Total Orders / Total Revenue

	​


In [9]:
# Revenue generated per order
product_features["revenue_per_order"] = (
    product_features["total_revenue"]
    / product_features["total_orders"]
)

In [10]:
product_features["revenue_per_order"].isna().sum()

np.int64(0)

In [11]:

np.isinf(product_features["revenue_per_order"]).sum()

np.int64(0)

In [12]:
product_features["revenue_per_order"].describe()

count    32951.000000
mean       154.900335
std        269.063064
min          0.850000
25%         42.900000
50%         84.900000
75%        163.816667
max      13440.000000
Name: revenue_per_order, dtype: float64

In [13]:
product_features[
    [
        "total_revenue",
        "total_orders",
        "revenue_per_order"
    ]
].head(10)

,total_revenue,total_orders,revenue_per_order
0,101.65,1,101.650000
1,129.90,1,129.900000
2,229.00,1,229.000000
3,117.80,2,58.900000
4,199.00,1,199.000000
5,52.00,1,52.000000
6,498.00,2,249.000000
7,350.10,7,50.014286
8,78.90,1,78.900000
9,489.86,12,40.821667


### 2. Revenue per Unit

Revenue per Unit = Total Units Sold / Total Revenue​

In [14]:
product_features["revenue_per_unit"] = (
    product_features["total_revenue"]
    / product_features["total_units_sold"]
)

In [15]:
product_features["revenue_per_unit"].isna().sum()

np.int64(0)

In [16]:
np.isinf(product_features["revenue_per_unit"]).sum()

np.int64(0)

In [17]:
product_features["revenue_per_unit"].describe()

count    32951.000000
mean       145.302464
std        246.895756
min          0.850000
25%         39.900000
50%         79.000000
75%        154.900000
max       6735.000000
Name: revenue_per_unit, dtype: float64

In [19]:
product_features[
    [
        "total_revenue",
        "total_units_sold",
        "revenue_per_unit",
        "avg_price"
    ]
].head(10)

,total_revenue,total_units_sold,revenue_per_unit,avg_price
0,101.65,1,101.65,101.65
1,129.90,1,129.90,129.90
2,229.00,1,229.00,229.00
3,117.80,2,58.90,58.90
4,199.00,1,199.00,199.00
5,52.00,1,52.00,52.00
6,498.00,2,249.00,249.00
7,350.10,9,38.90,38.90
8,78.90,1,78.90,78.90
9,489.86,14,34.99,34.99


In [20]:
product_features[
    ["avg_price", "revenue_per_unit"]
].corr()

,avg_price,revenue_per_unit
avg_price,1.0,1.0
revenue_per_unit,1.0,1.0


Since, the correlation between avg price and revenue per unit is 1, so it adds nothing value to the data so drop it

In [23]:
product_features = product_features.drop(columns = "revenue_per_unit")

### 3. Freight Ratio

In [25]:
product_features["freight_ratio"] = (
    product_features["total_freight"]
    / product_features["total_revenue"]
)

In [26]:
product_features["freight_ratio"].isna().sum()

np.int64(0)

In [27]:

np.isinf(product_features["freight_ratio"]).sum()

np.int64(0)

In [28]:
product_features[
    [
        "total_revenue",
        "total_freight",
        "freight_ratio"
    ]
].head()

,total_revenue,total_freight,freight_ratio
0,101.65,18.59,0.182882
1,129.90,13.93,0.107236
2,229.00,13.10,0.057205
3,117.80,39.20,0.332767
4,199.00,19.27,0.096834


### 4. Popularity score

In [ ]:
product_features["popularity_score"] = (
    product_features["total_orders"].rank(pct=True) * 0.5
    +
    product_features["total_units_sold"].rank(pct=True) * 0.5
)

In [36]:
product_features["popularity_score"].describe()

count    32951.000000
mean         0.500015
std          0.255406
min          0.285902
25%          0.285902
50%          0.285902
75%          0.721625
max          0.999985
Name: popularity_score, dtype: float64

In [37]:
product_features[
    [
        "total_orders",
        "total_units_sold",
        "popularity_score"
    ]
].sort_values(
    "popularity_score",
    ascending=False
).head(10)

,total_orders,total_units_sold,popularity_score
19742,467,488,0.999985
22112,431,527,0.999985
8613,352,484,0.999939
7364,311,392,0.999894
27039,323,343,0.999863
7079,291,388,0.999848
10840,287,373,0.999818
10867,306,323,0.999818
2794,269,281,0.999757
5692,259,260,0.999712


In [32]:
product_features.drop(
    columns=["orders_norm", "units_norm"],
    inplace=True
)

In [33]:
product_features[
    [
        "total_orders",
        "total_units_sold",
        "total_revenue",
        "freight_ratio",
        "popularity_score"
    ]
].describe()

,total_orders,total_units_sold,total_revenue,freight_ratio,popularity_score
count,32951.000000,32951.000000,32951.000000,32951.000000,32951.000000
mean,3.108403,3.418713,412.480462,0.319779,0.004561
std,9.456937,10.619709,1371.945598,0.353966,0.020187
min,1.000000,1.000000,2.200000,0.000088,0.000000
25%,1.000000,1.000000,59.900000,0.125566,0.000000
50%,1.000000,1.000000,136.750000,0.226778,0.000000
75%,2.000000,3.000000,329.000000,0.395137,0.003802
max,467.000000,527.000000,63885.000000,23.043137,0.962928


### 5. Premium product

In [40]:
premium_threshold = product_features["avg_price"].quantile(0.80)

product_features["premium_product"] = (
    product_features["avg_price"] >= premium_threshold
).astype(int)

print("Premium price threshold:", premium_threshold)
product_features["premium_product"].value_counts()

Premium price threshold: 179.99


premium_product
0    26325
1     6626
Name: count, dtype: int64

### 6. Best Seller Flag
Identifies products with exceptional demand.

In [41]:
bestseller_threshold = product_features["total_units_sold"].quantile(0.90)

product_features["bestseller_flag"] = (
    product_features["total_units_sold"] >= bestseller_threshold
).astype(int)

print("Bestseller threshold:", bestseller_threshold)
product_features["bestseller_flag"].value_counts()

Bestseller threshold: 6.0


bestseller_flag
0    29113
1     3838
Name: count, dtype: int64

### 7. High Rating product

Marks products with consistently strong customer satisfaction.

In [42]:
product_features["high_rating_product"] = (
    product_features["avg_review_score"] >= 4.5
).astype(int)

product_features["high_rating_product"].value_counts(dropna=False)

high_rating_product
1    16915
0    16036
Name: count, dtype: int64

In [43]:
product_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   product_id             32951 non-null  str    
 1   product_category_name  32341 non-null  str    
 2   total_units_sold       32951 non-null  int64  
 3   total_orders           32951 non-null  int64  
 4   total_revenue          32951 non-null  float64
 5   total_freight          32951 non-null  float64
 6   avg_price              32951 non-null  float64
 7   avg_review_score       32789 non-null  float64
 8   avg_delivery_days      32214 non-null  float64
 9   revenue_per_order      32951 non-null  float64
 10  freight_ratio          32951 non-null  float64
 11  popularity_score       32951 non-null  float64
 12  premium_product        32951 non-null  int64  
 13  bestseller_flag        32951 non-null  int64  
 14  high_rating_product    32951 non-null  int64  
dtypes: float64(8)

In [44]:
product_features.to_csv("../../data/processed/product_features.csv", index=False)
